# Module 38.1 — Image Classification

---

## 1. What is Image Classification?

Image classification is a supervised learning task in which a model analyses an input image and predicts which predefined class or category it belongs to.

### Examples

- Medical imaging: Normal vs. abnormal X-rays.
- Manufacturing: Defective vs. non-defective components.
- Agriculture: Healthy vs. diseased leaves.
- Autonomous vehicles: Identifying traffic signs.
- General applications: Classifying cats, dogs, cars, and other objects.

### Visual References

![CIFAR-10 Classification](https://images.openai.com/static-rsc-4/g_5YP7ZxOmQvy5GlyTEAzQy2lekCiPvuNNLXbN-IBp_gJOLyYKEZOuadpweCU7WDpfRZJ_xS_vajM8J7-EWZ4dn6U3mI7NoAExcbBlQ7LaZlNWtBVrm71F8y3KbO3RbD7C8W6tvnuGDX0YI0lq5MUr5OpF12c2aU1Q4_mLG4xyc?purpose=inline)

![Machine Vision](https://images.openai.com/static-rsc-4/2y3f550vAi8yubuIOWK0EtSxp3qP1zXdTpLEYoZv9_O_lp7QfFJi4Zkq7yGKoukf4RKhfpvPCsXu6Gab2Npii0oFCUlzNklNGiFS2Ie_gYhpk6Iw0e-YXs9WBAZLz6rLmUFs5026AkWIWHthlmitowIkgzI7rFK2EC8ts6aifZ0?purpose=inline)

### Image Classification vs. Other Computer Vision Tasks

| Task | What does the model predict? |
|---|---|
| Image Classification | What is present in the image? |
| Object Detection | What objects are present and where are they? |
| Image Segmentation | Which pixels belong to each object or region? |

**Example: Given an image of a dog**

- Classification → `Dog`
- Detection → `Dog` + Bounding Box
- Segmentation → Pixel-level Dog Mask

---

## 2. Classes and Labels

A **class** represents a category that the model must recognise.

A **label** represents the target assigned to an individual training image.

### Example Dataset

| Image | Class | Label |
|---|---|---|
| cat_01.jpg | Cat | 0 |
| dog_01.jpg | Dog | 1 |
| horse_01.jpg | Horse | 2 |
| cow_01.jpg | Cow | 3 |

The model internally works with numerical class indices.

```python
class_to_idx = {
    "cat": 0,
    "dog": 1,
    "horse": 2,
    "cow": 3
}

idx_to_class = {
    value: key for key, value in class_to_idx.items()
}
```

### Single-Label vs. Multi-Label Classification

**Single-label classification:** Each image belongs to exactly one class.

Example: An image is classified as either cat, dog, or horse.

**Multi-label classification:** An image can contain multiple applicable labels.

Example: An image may contain both a dog and a car.

| Property | Single-label | Multi-label |
|---|---|---|
| Output Activation | Softmax | Sigmoid |
| Typical Loss | CrossEntropyLoss | BCEWithLogitsLoss |
| Target Representation | Class Index | Multi-hot Vector |

For single-label classification, softmax converts logits into class probabilities:

$$
P(y=i\mid x)=\frac{e^{z_i}}{\sum_j e^{z_j}}
$$

For multi-label classification, sigmoid independently estimates the probability of each label:

$$
\sigma(z)=\frac{1}{1+e^{-z}}
$$

> **Important:** PyTorch's `CrossEntropyLoss` expects raw logits, not softmax probabilities.

---

## 3. Image Tensors

Deep learning models do not directly process image files such as JPG or PNG. Images must be converted into numerical tensors.

A colour image generally contains three channels:

- Red (R)
- Green (G)
- Blue (B)

### Image Dimensions

An image with a height of 224 pixels and width of 224 pixels has:

$$
H=224,\quad W=224,\quad C=3
$$

### Common Image Formats

| Format | Shape |
|---|---|
| HWC | `(224, 224, 3)` |
| CHW | `(3, 224, 224)` |
| BCHW | `(B, 3, 224, 224)` |

PyTorch CNN models normally expect the **BCHW** format.

Where:

- B = Batch Size
- C = Number of Channels
- H = Height
- W = Width

### Converting an Image into a Tensor

```python
from PIL import Image
from torchvision import transforms

image = Image.open("sample.jpg").convert("RGB")

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

image_tensor = transform(image)

print(image_tensor.shape)
```

**Output:**

```text
torch.Size([3, 224, 224])
```

### Adding a Batch Dimension

```python
image_tensor = image_tensor.unsqueeze(0)

print(image_tensor.shape)
```

**Output:**

```text
torch.Size([1, 3, 224, 224])
```

### Normalization

Normalization adjusts pixel values to a suitable numerical range.

`ToTensor()` converts standard 8-bit image pixel values from `[0, 255]` into floating-point values in `[0, 1]`.

Standard normalization:

$$
x_{\text{normalized}}=\frac{x-\mu}{\sigma}
$$

Where:

- $x$ = Original pixel value
- $\mu$ = Mean
- $\sigma$ = Standard deviation

For pretrained models, use the normalization values expected by the model's training configuration.

---

## 4. Classification Head

A classification model commonly consists of two major components:

1. **Feature Extractor (Backbone):** Extracts useful visual features from the input image.
2. **Classification Head:** Uses those features to predict the class.

### How Does It Work?

Consider an image classification model trained to identify four categories.

**Step 1 — Input Image**

The model receives an image tensor:

```python
[1, 3, 224, 224]
```

**Step 2 — Feature Extraction**

The backbone extracts visual information such as:

- Edges and textures.
- Shapes and patterns.
- Object parts and higher-level features.

**Step 3 — Classification Head**

The extracted feature vector is passed into a fully connected layer.

```python
import torch.nn as nn

classifier = nn.Linear(512, 4)
```

Here:

- 512 = Number of input features.
- 4 = Number of output classes.

**Step 4 — Logits**

The classification head produces four raw scores:

```python
import torch

logits = torch.tensor([[2.1, 0.5, 3.8, 0.2]])
```

These values are called **logits**.

**Step 5 — Prediction**

The highest logit corresponds to the predicted class.

```python
predicted_class = logits.argmax(dim=1)

print(predicted_class)
```

The predicted class index is `2`.

To obtain probabilities for a single-label classification task:

```python
probabilities = torch.softmax(logits, dim=1)
```

### Replacing the Classification Head in a Pretrained Model

Suppose you use ResNet18, whose original classification head predicts 1,000 ImageNet classes.

You want to classify four custom categories.

```python
import torch.nn as nn
from torchvision import models

model = models.resnet18(weights="DEFAULT")

model.fc = nn.Linear(
    model.fc.in_features,
    4
)
```

The backbone extracts features, while the replaced `fc` layer predicts your four custom classes.

---

## 5. Image Classification Training Pipeline

A typical image classification workflow:

### Step 1 — Dataset Preparation

Organise images into class-specific folders.

```text
dataset/
│
├── train/
│   ├── cat/
│   ├── dog/
│   └── horse/
│
├── val/
│   ├── cat/
│   ├── dog/
│   └── horse/
│
└── test/
    ├── cat/
    ├── dog/
    └── horse/
```

### Step 2 — Preprocessing

Typical preprocessing operations:

- Resize images to a consistent input size.
- Convert images into tensors.
- Normalize pixel values.
- Apply suitable transformations.

### Step 3 — Data Loading

PyTorch provides `Dataset` and `DataLoader` to organise and load batches of images.

```python
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

train_dataset = datasets.ImageFolder(
    "dataset/train",
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)
```

### Step 4 — Forward Pass

The input batch passes through the model to generate logits.

```python
outputs = model(images)
```

### Step 5 — Loss Calculation

For single-label multi-class classification:

```python
criterion = nn.CrossEntropyLoss()

loss = criterion(outputs, labels)
```

The loss measures the difference between predicted logits and the actual class labels.

### Step 6 — Backpropagation and Optimisation

```python
optimizer.zero_grad()

loss.backward()

optimizer.step()
```

This updates the trainable model parameters to reduce the loss.

### Step 7 — Validation

Evaluate the model on validation data to monitor generalisation.

```python
model.eval()

with torch.no_grad():
    for images, labels in val_loader:
        outputs = model(images)
```

### Important Training and Inference Modes

| Method | Purpose |
|---|---|
| `model.train()` | Enables training behaviour, including training-mode BatchNorm and Dropout |
| `model.eval()` | Switches modules to evaluation behaviour |
| `torch.no_grad()` | Disables gradient calculation |
| `torch.inference_mode()` | Provides an inference-oriented context with additional overhead reduction |

> **Important distinction:** `model.eval()` does not disable gradient calculation by itself. Use it together with `torch.no_grad()` or `torch.inference_mode()` when appropriate.

---

## 6. Image Classification Inference Pipeline

Inference means using a trained model to predict the class of a previously unseen image.

### Example

```python
import torch
from PIL import Image
from torchvision import transforms

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)
model.eval()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

image = Image.open("test_image.jpg").convert("RGB")

image_tensor = transform(image).unsqueeze(0).to(device)

with torch.inference_mode():
    logits = model(image_tensor)

    probabilities = torch.softmax(logits, dim=1)

predicted_idx = probabilities.argmax(dim=1).item()

print("Predicted class:", idx_to_class[predicted_idx])

print("Confidence:", probabilities[0, predicted_idx].item())
```

### GPU Verification

```python
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
```

Remember that the inference preprocessing must be compatible with the preprocessing used for the model's training, particularly when using pretrained weights.

---

## 7. Industry Applications

| Industry | Image Classification Application |
|---|---|
| Manufacturing | Defective vs. non-defective components |
| Healthcare | Medical image categorisation |
| Agriculture | Crop disease classification |
| Automotive | Traffic sign recognition |
| Retail | Product category recognition |
| Security | Image-based event categorisation |

### Practical Engineering Example: Manufacturing Defect Classification

Imagine an inspection system that classifies manufactured motor components.

Possible classes:

```python
classes = [
    "normal",
    "surface_defect",
    "crack",
    "deformation"
]
```

A camera captures the component, the model processes the image, and the classification output helps the quality inspection system flag suspicious parts.

**Engineering Consideration:** Classification alone does not identify the exact defect location. If the inspection system needs to locate the defect, object detection or segmentation may be more suitable.

---

## 8. Must Know / Good to Know / Advanced

| Priority | Concepts |
|---|---|
| Must Know | Classes, labels, image tensors, BCHW format, classification head, logits, softmax, CrossEntropyLoss, training and inference |
| Good to Know | Single-label vs. multi-label classification, normalization, pretrained model head replacement, GPU inference |
| Advanced | Calibration, confidence thresholds, class-specific decision thresholds, model deployment optimisation |

---

## What You Should Remember 🧠

1. Image classification predicts a category for an input image.
2. A class is a category; a label is the target assigned to an image.
3. PyTorch image models generally expect input tensors in `[B, C, H, W]` format.
4. A backbone extracts features, while a classification head predicts classes.
5. `CrossEntropyLoss` expects raw logits for single-label multi-class classification.
6. `model.eval()` and `torch.inference_mode()` are important during inference.
7. Training and inference preprocessing must be consistent.

---
